In [0]:
class Bronze_pit_stops():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        from pyspark.sql.types import IntegerType, StringType, DateType, StructType, StructField
        pit_stops_schema = StructType([
            StructField("raceId", IntegerType(), False),           # NOT NULL
            StructField("driverId", IntegerType(), False),         # NOT NULL
            StructField("stop", IntegerType(), False),             # NOT NULL
            StructField("lap", IntegerType(), False),              # NOT NULL
            StructField("time", StringType(), False),              # NOT NULL (time format like "13:52:25")
            StructField("duration", StringType(), True),           # Nullable
            StructField("milliseconds", IntegerType(), True)       # Nullable
        ])
        return pit_stops_schema

    def read_data(self):
        df= (spark.readStream
             .format("json")
             .schema(self.get_schema())
             .option('multiline','true')
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 2000)
             .load(f"{self.main_path}/formula1_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        from pyspark.sql.functions import current_timestamp,lit
        print(f"\nStarting Bronze pit_stops Stream...", end='')
        readDF = self.read_data()
        readDF= (readDF.withColumn('PitStopsIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-pit_stop")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('formula1_race.bronze.pit_stops') 
                    ) 
        print("Done")
        return sQuery   


In [0]:
from pyspark.sql.streaming import StreamingQueryListener

class MyListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"Query started: {event.id}")

    def onQueryProgress(self, event):
        print(f"Batch ID: {event.progress['batchId']}")
        print(f"Rows Read: {event.progress['numInputRows']}")
        print(f"Duration (ms): {event.progress['durationMs']}")
        print(f"Rows/sec: {event.progress['processedRowsPerSecond']}")

    def onQueryTerminated(self, event):
        print(f"Query terminated: {event.id}")

spark.streams.addListener(MyListener())

In [0]:
source='Ergast API'
Bronze_pit_stops_instance = Bronze_pit_stops("pit_stops",source)
Squery_Bronze_pit_stops =Bronze_pit_stops_instance.process()
Squery_Bronze_pit_stops.awaitTermination()
print("Successfully bronze-ingestion-pit_stops stream in running")
Squery_Bronze_pit_stops.stop()